In [1]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn matplotlib optuna plotly nbformat

In [2]:
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline

import optuna

from utils.constants import *

In [3]:
df = pd.read_csv("../data/3_gold/dataset-processed-gb.csv")

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [4]:
FIXED_PARAMS = {
    'max_iter': 2000,
    'early_stopping': True,
    'n_iter_no_change': 5,
    'tol': 1e-7,
    'categorical_features': categorical_features,
    'class_weight': 'balanced',
    'loss': 'log_loss',
    'random_state': RANDOM_STATE
}

In [5]:
# def get_pipeline(params, class_counts=None):
#     steps = []

#     if class_counts:
#         n_low, n_alarm, n_severe = class_counts[0], class_counts[1], class_counts[2]
#         steps.append(('under', RandomUnderSampler(sampling_strategy={0: max(n_alarm, n_low // 2)}, random_state=RANDOM_STATE)))
#         steps.append(('over', SMOTENC(categorical_features=categorical_features, sampling_strategy={1: max(n_severe, n_alarm // 2)}, random_state=RANDOM_STATE)))
    
#     steps.append(('classifier', HistGradientBoostingClassifier(**params)))
#     return Pipeline(steps=steps)


def train_on_folds(params):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        # clf = get_pipeline(params)

        clf = HistGradientBoostingClassifier(**params)

        try:
            clf.fit(X_train, y_train, X_val=X_valid, y_val=y_valid)
            preds = clf.predict(X_valid)
            f1 = f1_score(y_valid, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0
        
    return np.mean(scores), np.std(scores)


def objective(trial: optuna.trial.Trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 10, step=1),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 100, step=5),
        'l2_regularization': trial.suggest_float('l2_regularization',0, 1, step=0.1),
        **FIXED_PARAMS
    }
    avg, _ = train_on_folds(params)
    return avg

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

# Best F1: 0.5201657174578985
# Best Params: {'learning_rate': 0.03620016665032439, 'max_depth': 10, 'min_samples_leaf': 20, 'l2_regularization': 0.5}

[I 2026-02-06 02:06:46,455] A new study created in memory with name: no-name-f0661672-6172-4764-88a7-6e22fcc83963


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-06 03:50:43,411] Trial 5 finished with value: 0.5201657174578985 and parameters: {'learning_rate': 0.03620016665032439, 'max_depth': 10, 'min_samples_leaf': 20, 'l2_regularization': 0.5}. Best is trial 5 with value: 0.5201657174578985.
[I 2026-02-06 03:51:25,715] Trial 7 finished with value: 0.5151938567630103 and parameters: {'learning_rate': 0.02476751198107167, 'max_depth': 7, 'min_samples_leaf': 35, 'l2_regularization': 0.1}. Best is trial 5 with value: 0.5201657174578985.
[I 2026-02-06 04:07:05,081] Trial 2 finished with value: 0.510318924783866 and parameters: {'learning_rate': 0.00998603917390668, 'max_depth': 9, 'min_samples_leaf': 80, 'l2_regularization': 0.2}. Best is trial 5 with value: 0.5201657174578985.
[I 2026-02-06 04:10:45,357] Trial 6 finished with value: 0.5062166795262946 and parameters: {'learning_rate': 0.005670674901806588, 'max_depth': 6, 'min_samples_leaf': 75, 'l2_regularization': 0.1}. Best is trial 5 with value: 0.5201657174578985.
[I 2026-02-06 0

In [7]:
import pickle
from pathlib import Path

output_dir = Path('../results/optimization/histgb')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

In [8]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [9]:
avg_f1_final, std_f1_final = train_on_folds(best_trial.params)

In [11]:
print(avg_f1_final)
print(std_f1_final)

0.48136409327801477
0.0028036089088351863
